# 11. Code Interpreter with Image Output

This notebook demonstrates how to generate and retrieve images/charts from code interpreter.

In [ ]:
# Install OpenAI package
!pip install openai matplotlib

In [ ]:
from openai import OpenAI
import os
import platform
import json
import sys

In [ ]:
# 🔐 ENTER YOUR API KEY HERE
OPENAI_API_KEY = "YOUR_API_KEY_HERE"

client = OpenAI(api_key=OPENAI_API_KEY)

In [ ]:
# Create output directory
output_dir = "maersk_outputs"
os.makedirs(output_dir, exist_ok=True)

print("🔧 Investment Calculator with gpt-4o")
print("=" * 60)

In [ ]:
# Send request to LLM
response = client.responses.create(
    model="gpt-4o-mini",
    tools=[
        {
            "type": "code_interpreter",
            "container": {"type": "auto"}
        }
    ],
    input="""
Calculate simple interest and create a bar chart.

Principal: $1000
Rate: 10% per annum
Time: 10 years

Write Python code to:
1. Calculate the interest values
2. Create a matplotlib bar chart
3. Save it as a PNG file
4. Return the image file

Execute the code using the code interpreter.
""",
    max_output_tokens=2000,
)

print("\n✅ Model Response:")
print(response.output_text)

In [ ]:
print("\n📁 Searching for generated files...")

files_found = False

# Parse response output safely
if hasattr(response, "output"):
    for item in response.output:
        item_dict = item.model_dump()

        # Check if this is a message type with content
        if item_dict.get("type") == "message":
            content_items = item_dict.get("content", [])
            
            for content in content_items:
                annotations = content.get("annotations", [])
                
                for annotation in annotations:
                    if annotation.get("type") == "container_file_citation":
                        files_found = True
                        file_id = annotation.get("file_id")
                        filename = annotation.get("filename", f"{file_id}.png")
                        container_id = annotation.get("container_id")

                        print(f"✅ Found file: {filename}")
                        print(f"   Container ID: {container_id}")
                        print(f"   File ID: {file_id}")

                        try:
                            print(f"   Downloading file content...")
                            
                            # Use the Files API to get the content
                            file_content = client.files.content(file_id)
                            
                            file_path = os.path.join(output_dir, filename)

                            with open(file_path, "wb") as f:
                                f.write(file_content.content)

                            print(f"💾 Saved file to: {file_path}")

                        except Exception as e:
                            print(f"❌ Error downloading file: {e}")

# If no files found, save debug info
if not files_found:
    print("⚠️ No files returned by model. Saving debug info...")
    debug_path = os.path.join(output_dir, "debug_response.json")
    with open(debug_path, "w") as f:
        json.dump(response.model_dump(), f, indent=2, default=str)
    print(f"📄 Debug saved to: {debug_path}")

print("\n" + "=" * 60)
print("✅ Script finished")

In [ ]:
# Display images in Colab
from IPython.display import Image, display
import glob

# Find all PNG files in the output directory
image_files = glob.glob(os.path.join(output_dir, "*.png"))

if image_files:
    print("\n📊 Generated Images:")
    for img_path in image_files:
        print(f"\n{os.path.basename(img_path)}:")
        display(Image(img_path))
else:
    print("No images found to display")